# 00 - Bootstrap the Fabric Lakehouse

Creates the control-plane, attempt-output, reconciliation, benchmark, and gold Delta tables plus committed-output views. Run once per environment and again only for reviewed, backward-compatible schema releases.

An empty new Lakehouse can be bootstrapped directly. If any target tables already exist, set `CONFIRM_WRITERS_STOPPED=True` only for an exclusive maintenance window after stopping all application writers, workers, and queued jobs. Keep the saved default `False`. Durable active work states left by a definitively stopped worker do not block schema migration: preserve these rows, run bootstrap first, then run the updated watchdog to recover them before resuming admission. Maintenance and reset still require recovery before running.

Bootstrap seeds exactly one unowned `global` control-writer row without overwriting an existing owner. The owner does not expire: a failed or ambiguous mutation intentionally fails closed. A retained owner requires separate explicit offline recovery after proving the writer stopped; this notebook never steals or clears it.

**After importing into Fabric:** On the configuration code cell, select **... -> Toggle parameter cell** and confirm the parameter indicator. Then attach and pin `people_counter_<environment>` as this notebook's default Lakehouse.

In [ ]:
DATABASE = ""
TABLE_PREFIX = "people_counter"
CONFIRM_WRITERS_STOPPED = False

In [ ]:
from datetime import datetime, timezone
import re

from pyspark.sql import SparkSession, Window, functions as F


IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def identifier(value: str, name: str, *, allow_empty: bool = False) -> str:
    if allow_empty and not value:
        return ""
    if IDENTIFIER.fullmatch(value) is None:
        raise ValueError(f"{name} is not a valid SQL identifier: {value!r}")
    return value


def parameter_bool(value: object, name: str) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, str) and value.strip().lower() in {"true", "false"}:
        return value.strip().lower() == "true"
    raise ValueError(f"{name} must be true or false")


writers_stopped = parameter_bool(CONFIRM_WRITERS_STOPPED, "CONFIRM_WRITERS_STOPPED")
database = identifier(DATABASE.strip(), "DATABASE", allow_empty=True)
prefix = identifier(TABLE_PREFIX.strip(), "TABLE_PREFIX")
spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("Attach a Fabric Lakehouse and start a Spark session")
spark_session = spark_candidate
spark_session.conf.set("spark.sql.session.timeZone", "UTC")

def name(suffix: str) -> str:
    table = f"{prefix}_{suffix}"
    return f"`{database}`.`{table}`" if database else f"`{table}`"


def storage_name(suffix: str) -> str:
    table = f"{prefix}_{suffix}"
    return f"{database}.{table}" if database else table


ddl = {
    "control_writer": """
        lock_name STRING,
        owner_id STRING,
        acquired_at TIMESTAMP
    """,
    "worker_events": """
        event_id STRING,
        work_id STRING,
        attempt_id STRING,
        worker_execution_id STRING,
        capture_date DATE,
        sequence BIGINT,
        event_kind STRING,
        payload_json STRING,
        created_at TIMESTAMP
    """,
    "worker_event_receipts": """
        event_id STRING,
        work_id STRING,
        attempt_id STRING,
        worker_execution_id STRING,
        sequence BIGINT,
        outcome STRING,
        message STRING,
        applied_at TIMESTAMP
    """,
    "event_receipts": """
        event_key STRING NOT NULL,
        event_source STRING NOT NULL,
        event_id STRING NOT NULL,
        event_type STRING NOT NULL,
        event_time TIMESTAMP NOT NULL,
        subject STRING NOT NULL,
        manifest_uri STRING NOT NULL,
        work_id STRING,
        received_at TIMESTAMP NOT NULL,
        registration_status STRING NOT NULL,
        pipeline_run_id STRING,
        error_type STRING,
        error_message STRING
    """,
    "video_work": """
        work_id STRING NOT NULL,
        asset_id STRING NOT NULL,
        asset_version STRING NOT NULL,
        source_uri STRING NOT NULL,
        manifest_uri STRING NOT NULL,
        source_etag STRING NOT NULL,
        expected_size_bytes BIGINT NOT NULL,
        expected_sha256 STRING,
        camera_id STRING NOT NULL,
        location_id STRING NOT NULL,
        captured_at_utc TIMESTAMP NOT NULL,
        camera_timezone STRING NOT NULL,
        duration_seconds DOUBLE,
        priority INT NOT NULL,
        status STRING NOT NULL,
        received_at TIMESTAMP NOT NULL,
        queued_at TIMESTAMP NOT NULL,
        queue_entered_at TIMESTAMP NOT NULL,
        not_before_at TIMESTAMP,
        attempt_count INT NOT NULL,
        max_attempts INT NOT NULL,
        lease_owner_attempt_id STRING,
        lease_dispatcher_id STRING,
        lease_acquired_at TIMESTAMP,
        lease_expires_at TIMESTAMP,
        last_heartbeat_at TIMESTAMP,
        committed_attempt_id STRING,
        completed_at TIMESTAMP,
        last_error_category STRING,
        last_error_type STRING,
        last_error_message STRING,
        last_replay_id STRING,
        replay_generation BIGINT NOT NULL,
        config_json STRING NOT NULL,
        config_sha256 STRING NOT NULL,
        runtime_sha256 STRING NOT NULL,
        capture_date DATE NOT NULL
    """,
    "video_attempts": """
        attempt_id STRING NOT NULL,
        work_id STRING NOT NULL,
        dispatcher_id STRING NOT NULL,
        pipeline_run_id STRING,
        activity_run_id STRING,
        fabric_job_instance_id STRING,
        worker_execution_id STRING,
        sdk_version STRING,
        bundle_manifest_sha256 STRING,
        config_sha256 STRING NOT NULL,
        status STRING NOT NULL,
        claimed_at TIMESTAMP NOT NULL,
        staging_started_at TIMESTAMP,
        inference_started_at TIMESTAMP,
        writing_started_at TIMESTAMP,
        completed_at TIMESTAMP,
        last_heartbeat_at TIMESTAMP,
        input_sha256 STRING,
        source_size_bytes BIGINT,
        source_duration_seconds DOUBLE,
        source_fps DOUBLE,
        total_source_frames BIGINT,
        processed_frames BIGINT,
        effective_sample_fps DOUBLE,
        processing_seconds DOUBLE,
        distinct_people BIGINT,
        line_in_count BIGINT,
        line_out_count BIGINT,
        retryable BOOLEAN,
        error_category STRING,
        error_type STRING,
        error_message STRING,
        capture_date DATE NOT NULL
    """,
    "dispatcher_leases": """
        lock_name STRING NOT NULL,
        owner_id STRING NOT NULL,
        acquired_at TIMESTAMP NOT NULL,
        expires_at TIMESTAMP NOT NULL
    """,
    "registration_leases": """
        lock_name STRING NOT NULL,
        owner_id STRING NOT NULL,
        acquired_at TIMESTAMP NOT NULL,
        expires_at TIMESTAMP NOT NULL
    """,
    "replay_requests": """
        replay_id STRING NOT NULL,
        work_id STRING NOT NULL,
        requested_by STRING NOT NULL,
        reason STRING NOT NULL,
        requested_at TIMESTAMP NOT NULL,
        previous_status STRING NOT NULL,
        replay_generation BIGINT NOT NULL,
        applied_at TIMESTAMP,
        capture_date DATE NOT NULL
    """,
    "telemetry_attempts": """
        work_id STRING NOT NULL,
        attempt_id STRING NOT NULL,
        camera_id STRING NOT NULL,
        location_id STRING NOT NULL,
        captured_at_utc TIMESTAMP NOT NULL,
        capture_date DATE NOT NULL,
        recorded_at TIMESTAMP NOT NULL,
        person_id BIGINT NOT NULL,
        entry_frame BIGINT NOT NULL,
        exit_frame BIGINT NOT NULL,
        entry_seconds DOUBLE NOT NULL,
        exit_seconds DOUBLE NOT NULL,
        person_entry_at_utc TIMESTAMP NOT NULL,
        person_exit_at_utc TIMESTAMP NOT NULL,
        duration_seconds DOUBLE NOT NULL
    """,
    "line_count_attempts": """
        work_id STRING NOT NULL,
        attempt_id STRING NOT NULL,
        camera_id STRING NOT NULL,
        location_id STRING NOT NULL,
        captured_at_utc TIMESTAMP NOT NULL,
        capture_date DATE NOT NULL,
        recorded_at TIMESTAMP NOT NULL,
        frame BIGINT NOT NULL,
        video_seconds STRING NOT NULL,
        video_timestamp STRING NOT NULL,
        observed_at_utc TIMESTAMP NOT NULL,
        frame_in_count BIGINT NOT NULL,
        frame_out_count BIGINT NOT NULL,
        cumulative_in_count BIGINT NOT NULL,
        cumulative_out_count BIGINT NOT NULL,
        line_start_x BIGINT NOT NULL,
        line_start_y BIGINT NOT NULL,
        line_end_x BIGINT NOT NULL,
        line_end_y BIGINT NOT NULL
    """,
    "reconciliation_findings": """
        finding_id STRING NOT NULL,
        detected_at TIMESTAMP NOT NULL,
        last_detected_at TIMESTAMP NOT NULL,
        severity STRING NOT NULL,
        finding_type STRING NOT NULL,
        work_id STRING,
        attempt_id STRING,
        details STRING NOT NULL,
        resolved_at TIMESTAMP,
        capture_date DATE
    """,
    "processing_benchmarks": """
        benchmark_id STRING NOT NULL,
        benchmark_batch_id STRING NOT NULL,
        benchmark_started_at TIMESTAMP NOT NULL,
        completed_at TIMESTAMP NOT NULL,
        capacity_sku STRING NOT NULL,
        runtime_version STRING NOT NULL,
        sdk_version STRING NOT NULL,
        config_sha256 STRING NOT NULL,
        sample_name STRING NOT NULL,
        video_duration_seconds DOUBLE NOT NULL,
        end_to_end_seconds DOUBLE NOT NULL,
        overhead_seconds DOUBLE NOT NULL,
        processing_seconds DOUBLE NOT NULL,
        source_stage_seconds DOUBLE,
        runtime_load_seconds DOUBLE,
        video_processing_seconds DOUBLE,
        result_persist_seconds DOUBLE,
        sampled_frames BIGINT,
        artifact_mode STRING,
        driver_cores INT,
        active_workers_per_driver INT,
        threads_per_worker INT,
        interop_threads_configured BOOLEAN,
        spark_application_id STRING,
        speed_x_realtime DOUBLE NOT NULL,
        peak_memory_mb DOUBLE,
        concurrent_workers INT NOT NULL,
        succeeded BOOLEAN NOT NULL,
        error_message STRING
    """,
    "gold_flow_minute": """
        minute_utc TIMESTAMP NOT NULL,
        time_key INT NOT NULL,
        camera_id STRING NOT NULL,
        location_id STRING NOT NULL,
        entries BIGINT NOT NULL,
        exits BIGINT NOT NULL,
        net_flow BIGINT NOT NULL,
        source_videos BIGINT NOT NULL,
        refreshed_at TIMESTAMP NOT NULL,
        flow_date DATE NOT NULL
    """,
    "gold_flow_hour": """
        hour_utc TIMESTAMP NOT NULL,
        time_key INT NOT NULL,
        camera_id STRING NOT NULL,
        location_id STRING NOT NULL,
        entries BIGINT NOT NULL,
        exits BIGINT NOT NULL,
        net_flow BIGINT NOT NULL,
        source_videos BIGINT NOT NULL,
        refreshed_at TIMESTAMP NOT NULL,
        flow_date DATE NOT NULL
    """,
    "gold_video": """
        work_id STRING NOT NULL,
        captured_at_utc TIMESTAMP NOT NULL,
        time_key INT NOT NULL,
        camera_id STRING NOT NULL,
        location_id STRING NOT NULL,
        config_sha256 STRING NOT NULL,
        video_duration_seconds DOUBLE,
        processing_seconds DOUBLE,
        speed_x_realtime DOUBLE,
        distinct_people BIGINT,
        line_in_count BIGINT,
        line_out_count BIGINT,
        completed_at TIMESTAMP NOT NULL,
        capture_date DATE NOT NULL
    """,
    "gold_operations_hour": """
        hour_utc TIMESTAMP NOT NULL,
        time_key INT NOT NULL,
        queued BIGINT NOT NULL,
        started BIGINT NOT NULL,
        succeeded BIGINT NOT NULL,
        failed BIGINT NOT NULL,
        deferred BIGINT NOT NULL,
        video_hours_completed DOUBLE NOT NULL,
        average_processing_seconds DOUBLE,
        p95_processing_seconds DOUBLE,
        refreshed_at TIMESTAMP NOT NULL,
        operation_date DATE NOT NULL
    """,
    "gold_dim_date": """
        date_key DATE NOT NULL,
        calendar_year INT NOT NULL,
        calendar_quarter INT NOT NULL,
        calendar_month INT NOT NULL,
        month_name STRING NOT NULL,
        month_short_name STRING NOT NULL,
        year_month STRING NOT NULL,
        day_of_month INT NOT NULL,
        iso_day_of_week INT NOT NULL,
        iso_week_year INT NOT NULL,
        iso_week_of_year INT NOT NULL,
        iso_year_week STRING NOT NULL,
        day_name STRING NOT NULL,
        is_weekend BOOLEAN NOT NULL,
        refreshed_at TIMESTAMP NOT NULL
    """,
    "gold_dim_time": """
        time_key INT NOT NULL,
        hour_24 INT NOT NULL,
        minute_of_hour INT NOT NULL,
        time_label STRING NOT NULL,
        hour_label STRING NOT NULL,
        day_part STRING NOT NULL,
        refreshed_at TIMESTAMP NOT NULL
    """,
    "gold_dim_camera": """
        camera_id STRING NOT NULL,
        location_id STRING NOT NULL,
        camera_timezone STRING NOT NULL,
        first_capture_utc TIMESTAMP NOT NULL,
        last_capture_utc TIMESTAMP NOT NULL,
        video_count BIGINT NOT NULL,
        refreshed_at TIMESTAMP NOT NULL
    """,
    "gold_dim_location": """
        location_id STRING NOT NULL,
        first_capture_utc TIMESTAMP NOT NULL,
        last_capture_utc TIMESTAMP NOT NULL,
        camera_count BIGINT NOT NULL,
        video_count BIGINT NOT NULL,
        refreshed_at TIMESTAMP NOT NULL
    """,
    "gold_dim_video": """
        work_id STRING NOT NULL,
        asset_id STRING NOT NULL,
        asset_version STRING NOT NULL,
        camera_id STRING NOT NULL,
        location_id STRING NOT NULL,
        captured_at_utc TIMESTAMP NOT NULL,
        capture_date DATE NOT NULL,
        time_key INT NOT NULL,
        camera_timezone STRING NOT NULL,
        config_sha256 STRING NOT NULL,
        refreshed_at TIMESTAMP NOT NULL
    """,
    "gold_dim_model_config": """
        config_sha256 STRING NOT NULL,
        config_json STRING NOT NULL,
        pipeline STRING NOT NULL,
        device_variant STRING NOT NULL,
        device STRING NOT NULL,
        batch_size INT NOT NULL,
        sample_fps DOUBLE,
        detection_threshold DOUBLE NOT NULL,
        use_fp16 BOOLEAN NOT NULL,
        detector_model STRING NOT NULL,
        camera_motion_compensation BOOLEAN,
        counting_line_json STRING NOT NULL,
        first_capture_utc TIMESTAMP NOT NULL,
        last_capture_utc TIMESTAMP NOT NULL,
        video_count BIGINT NOT NULL,
        refreshed_at TIMESTAMP NOT NULL
    """
}

partitioned = {
    "video_work": "capture_date",
    "video_attempts": "capture_date",
    "replay_requests": "capture_date",
    "telemetry_attempts": "capture_date",
    "line_count_attempts": "capture_date",
    "gold_flow_minute": "flow_date",
    "gold_flow_hour": "flow_date",
    "gold_video": "capture_date",
    "gold_operations_hour": "operation_date",
    "gold_dim_video": "capture_date",
}

existing_tables = [
    storage_name(suffix)
    for suffix in ddl
    if spark_session.catalog.tableExists(storage_name(suffix))
]
if existing_tables and not writers_stopped:
    raise RuntimeError(
        "Existing tables require CONFIRM_WRITERS_STOPPED=True in an exclusive offline maintenance window"
    )


def control_writer_seed_rows():
    rows = spark_session.table(storage_name("control_writer")).limit(2).collect()
    if rows and (len(rows) != 1 or rows[0].lock_name != "global"):
        raise RuntimeError("Invalid control_writer seed; explicit offline recovery is required")
    if rows and rows[0].owner_id is not None:
        raise RuntimeError(
            "control_writer has an owner; prove the writer stopped and perform explicit offline recovery"
        )
    return rows


if spark_session.catalog.tableExists(storage_name("control_writer")):
    control_writer_seed_rows()

if database:
    create_database_sql = "CREATE DATABASE IF NOT EXISTS `" + database + "`"
    spark_session.sql(create_database_sql)

for suffix, columns in ddl.items():
    # Fabric Runtime's Spark SQL parser does not accept column-level
    # NOT NULL constraints in this CREATE TABLE USING DELTA form.
    compatible_columns = columns.replace(" NOT NULL", "")
    empty = spark_session.createDataFrame([], schema=compatible_columns)
    table_writer = empty.write.format("delta").mode("ignore")
    if suffix in partitioned:
        table_writer = table_writer.partitionBy(partitioned[suffix])
    table_writer.saveAsTable(storage_name(suffix))

if not control_writer_seed_rows():
    control_seed = spark_session.createDataFrame(
        [("global", None, None)],
        schema=spark_session.table(storage_name("control_writer")).schema,
    )
    control_seed.write.format("delta").mode("append").saveAsTable(storage_name("control_writer"))
if len(control_writer_seed_rows()) != 1:
    raise RuntimeError("Bootstrap did not create exactly one unowned global control_writer row")

from people_counter.fabric_control import ControlWriter

writer = ControlWriter(spark_session, storage_name("control_writer"))
DeltaTable = writer.tables

In [ ]:
required_columns_by_table = {
    "video_work": {
        "last_replay_id": "STRING",
        "replay_generation": "BIGINT",
        "queue_entered_at": "TIMESTAMP",
        "runtime_sha256": "STRING",
    },
    "replay_requests": {"replay_generation": "BIGINT"},
    "video_attempts": {"worker_execution_id": "STRING"},
    "gold_flow_minute": {"time_key": "INT"},
    "gold_flow_hour": {"time_key": "INT"},
    "gold_video": {"time_key": "INT", "config_sha256": "STRING"},
    "gold_operations_hour": {"time_key": "INT", "deferred": "BIGINT"},
    "gold_dim_date": {"iso_week_year": "INT", "iso_year_week": "STRING"},
    "processing_benchmarks": {
        "source_stage_seconds": "DOUBLE",
        "runtime_load_seconds": "DOUBLE",
        "video_processing_seconds": "DOUBLE",
        "result_persist_seconds": "DOUBLE",
        "sampled_frames": "BIGINT",
        "artifact_mode": "STRING",
        "driver_cores": "INT",
        "active_workers_per_driver": "INT",
        "threads_per_worker": "INT",
        "interop_threads_configured": "BOOLEAN",
        "spark_application_id": "STRING",
    },
}
for suffix, required_columns in required_columns_by_table.items():
    existing_columns = set(spark_session.table(storage_name(suffix)).columns)
    for column, data_type in required_columns.items():
        if column not in existing_columns:
            writer.run(lambda: spark_session.sql(
                f"ALTER TABLE {name(suffix)} ADD COLUMNS (`{column}` {data_type})"
            ))

In [ ]:
DeltaTable.forName(spark_session, storage_name("video_work")).update(
    condition=F.col("queue_entered_at").isNull(),
    set={"queue_entered_at": F.col("queued_at")},
)
DeltaTable.forName(spark_session, storage_name("video_work")).update(
    condition=F.col("replay_generation").isNull(),
    set={"replay_generation": F.lit(0).cast("long")},
)

replay_requests_name = storage_name("replay_requests")
legacy_requests = spark_session.table(replay_requests_name).where(
    F.col("replay_generation").isNull()
)
if legacy_requests.head(1):
    if legacy_requests.where(F.col("applied_at").isNull()).head(1):
        raise RuntimeError(
            "Legacy unapplied replay requests require operator reconciliation before bootstrap"
        )
    affected_work = legacy_requests.select("work_id").distinct()
    mixed_requests = (
        spark_session.table(replay_requests_name)
        .where(F.col("replay_generation").isNotNull())
        .join(affected_work, "work_id", "inner")
    )
    if mixed_requests.head(1):
        raise RuntimeError("Mixed legacy and generated replay requests require operator reconciliation")
    incompatible_work = (
        spark_session.table(storage_name("video_work"))
        .join(affected_work, "work_id", "inner")
        .where(
            (F.coalesce(F.col("replay_generation"), F.lit(0)) != 0)
            | F.col("last_replay_id").isNotNull()
        )
    )
    if incompatible_work.head(1):
        raise RuntimeError("Legacy replay requests conflict with work replay generations")
    ascending = Window.partitionBy("work_id").orderBy("requested_at", "replay_id")
    assignments = legacy_requests.withColumn(
        "assigned_generation",
        F.row_number().over(ascending).cast("long"),
    )
    request_updates = assignments.select(
        "replay_id",
        "capture_date",
        "assigned_generation",
    )
    (
        DeltaTable.forName(spark_session, replay_requests_name)
        .alias("t")
        .merge(
            request_updates.alias("s"),
            "t.replay_id = s.replay_id AND t.capture_date = s.capture_date",
        )
        .whenMatchedUpdate(set={"replay_generation": "s.assigned_generation"})
        .execute()
    )
    descending = Window.partitionBy("work_id").orderBy(
        F.col("assigned_generation").desc()
    )
    latest = (
        assignments.withColumn("rank", F.row_number().over(descending))
        .where(F.col("rank") == 1)
        .select(
            "work_id",
            F.col("replay_id").alias("last_replay_id"),
            F.col("assigned_generation").alias("replay_generation"),
        )
    )
    (
        DeltaTable.forName(spark_session, storage_name("video_work"))
        .alias("t")
        .merge(latest.alias("s"), "t.work_id = s.work_id")
        .whenMatchedUpdate(
            set={
                "last_replay_id": "s.last_replay_id",
                "replay_generation": "s.replay_generation",
            }
        )
        .execute()
    )

In [ ]:
applied_replays = spark_session.table(storage_name("replay_requests")).where(
    F.col("applied_at").isNotNull() & F.col("replay_generation").isNotNull()
)
duplicate_generations = (
    applied_replays.groupBy("work_id", "replay_generation")
    .count()
    .where(F.col("count") != 1)
)
if duplicate_generations.head(1):
    raise RuntimeError("Duplicate replay generations require operator reconciliation")
latest_window = Window.partitionBy("work_id").orderBy(
    F.col("replay_generation").desc(),
    F.col("requested_at").desc(),
    F.col("replay_id").desc(),
)
latest_applied_replays = (
    applied_replays.withColumn("rank", F.row_number().over(latest_window))
    .where(F.col("rank") == 1)
    .select(
        "work_id",
        F.col("replay_id").alias("last_replay_id"),
        "replay_generation",
    )
)
replay_conflicts = (
    spark_session.table(storage_name("video_work")).alias("w")
    .join(latest_applied_replays.alias("r"), "work_id", "inner")
    .where(
        (F.coalesce(F.col("w.replay_generation"), F.lit(0)) == F.col("r.replay_generation"))
        & F.col("w.last_replay_id").isNotNull()
        & (F.col("w.last_replay_id") != F.col("r.last_replay_id"))
    )
)
if replay_conflicts.head(1):
    raise RuntimeError("Applied replay history conflicts with video_work replay identity")
(
    DeltaTable.forName(spark_session, storage_name("video_work"))
    .alias("t")
    .merge(latest_applied_replays.alias("s"), "t.work_id = s.work_id")
    .whenMatchedUpdate(
        condition=(
            "COALESCE(t.replay_generation, 0) < s.replay_generation OR "
            "(COALESCE(t.replay_generation, 0) = s.replay_generation AND t.last_replay_id IS NULL)"
        ),
        set={
            "last_replay_id": "s.last_replay_id",
            "replay_generation": "s.replay_generation",
        },
    )
    .execute()
)

In [ ]:
epoch = datetime(1970, 1, 1, tzinfo=timezone.utc)
registration_seed = spark_session.createDataFrame(
    [("global", "", epoch, epoch)],
    schema=spark_session.table(storage_name("registration_leases")).schema,
)
(
    DeltaTable.forName(spark_session, storage_name("registration_leases"))
    .alias("t")
    .merge(registration_seed.alias("s"), "t.lock_name = s.lock_name")
    .whenNotMatchedInsertAll()
    .execute()
)

telemetry_view_sql = (
    "CREATE OR REPLACE VIEW " + name("telemetry_committed") + " AS "
    "SELECT t.* FROM " + name("telemetry_attempts") + " t "
    "INNER JOIN " + name("video_work") + " w "
    "ON t.work_id = w.work_id "
    "AND t.attempt_id = w.committed_attempt_id "
    "WHERE w.status = 'SUCCEEDED'"
)
writer.run(lambda: spark_session.sql(telemetry_view_sql))

line_counts_view_sql = (
    "CREATE OR REPLACE VIEW " + name("line_counts_committed") + " AS "
    "SELECT l.* FROM " + name("line_count_attempts") + " l "
    "INNER JOIN " + name("video_work") + " w "
    "ON l.work_id = w.work_id "
    "AND l.attempt_id = w.committed_attempt_id "
    "WHERE w.status = 'SUCCEEDED'"
)
writer.run(lambda: spark_session.sql(line_counts_view_sql))

runs_view_sql = (
    "CREATE OR REPLACE VIEW " + name("runs_committed") + " AS "
    "SELECT w.work_id, w.asset_id, w.asset_version, w.camera_id, "
    "w.location_id, w.captured_at_utc, w.camera_timezone, "
    "w.duration_seconds, w.committed_attempt_id, w.completed_at, "
    "w.config_sha256, a.processing_seconds, a.effective_sample_fps, "
    "a.processed_frames, a.distinct_people, a.line_in_count, "
    "a.line_out_count, a.input_sha256 FROM " + name("video_work") + " w "
    "INNER JOIN " + name("video_attempts") + " a "
    "ON w.work_id = a.work_id "
    "AND w.committed_attempt_id = a.attempt_id "
    "WHERE w.status = 'SUCCEEDED'"
)
writer.run(lambda: spark_session.sql(runs_view_sql))

expected = {f"{prefix}_{suffix}" for suffix in ddl}
show_tables = "SHOW TABLES IN `" + database + "`" if database else "SHOW TABLES"
actual = {row.tableName for row in spark_session.sql(show_tables).collect()}
missing = sorted(expected - actual)
if missing:
    raise RuntimeError(f"Bootstrap did not create tables: {missing}")

print(f"Created or verified {len(expected)} Delta tables and 3 committed views")